In [ ]:
from __future__ import annotations

import importlib
import pickle
from pathlib import Path

import numpy as np
import pandas as pd

%cd /Users/apple/Desktop/Academics/Research_work/TheVirtueOfComplexity_PaperReplication_Experimentation-master

import ipca
import src
import src.grassmann_ipca_workflow as GrassmannIPCAWorkflowModule
import src.ipca_workflow as IPCAWorkflowModule
import src.portfolio_utils as portfolio_utils

importlib.reload(IPCAWorkflowModule)
importlib.reload(GrassmannIPCAWorkflowModule)
importlib.reload(portfolio_utils)
importlib.reload(ipca)


## How To Run This Notebook

1. Run the import cell, then run the configuration cell directly below it.
2. Set `USE_SYNTHETIC_DATA = True` if you want the synthetic panel. In that case, run the synthetic-data cell and skip the OpenAP / WRDS cells.
3. Set `USE_SYNTHETIC_DATA = False` if you want real data. Then either set `USE_CACHED_REAL_CHAR_DATA = True` and run the cached-parquet cell, or leave it `False` and run the OpenAP download / cleaning / WRDS merge cells.
4. Edit `RUN_CONFIGS` to choose the `(k, T)` experiments you want to compare.
5. Run the multi-config sweep cell. It saves one pickle per `(k, T)` run so you can reload later without rerunning the IPCA fits.
6. Run the export cell to write `features_data*.xlsx`, `portfolio_performance*.xlsx`, and `oos_r2*.xlsx` for every completed experiment.
7. After the exports finish, use the last plot cells to compare effective-rank diagnostics, portfolio Sharpe / drawdown, and OOS `R^2` across different `k` values.

Notes: the synthetic panel already includes `excess_ret`, `ret_adj`, `y_ipca`, and `mcap`, so you do not need the OpenAP or WRDS download cells for that path.

Notes: if you only want to plot previously finished runs, set `LOAD_RESULTS_FROM_PICKLES = True` and run the pickle-load cell instead of rerunning the sweep.


In [ ]:
WORKDIR = Path.cwd()
RESULTS_DIR = WORKDIR

USE_SYNTHETIC_DATA = True
USE_CACHED_REAL_CHAR_DATA = False
CHAR_DATA_CACHE_PATH = Path("/Users/apple/Desktop/Academics/char_data.parquet")

REAL_DATA_LABEL = "all_data"
SYNTHETIC_DATA_LABEL = "synthetic"
data_label = SYNTHETIC_DATA_LABEL if USE_SYNTHETIC_DATA else REAL_DATA_LABEL
PLOT_DATA_LABEL = data_label

REAL_CHAR_DOWNLOAD = {
    "start_yyyymm": "199001",
    "end_yyyymm": "202512",
}
REAL_RETURNS_DOWNLOAD = {
    "start_date": "1990-01-01",
    "end_date": "2026-01-31",
}

RUN_CONFIGS = [
    {"k": 12, "T": 24},
    {"k": 14, "T": 36},
    {"k": 16, "T": 24},
    {"k": 18, "T": 24},
    {"k": 18, "T": 36},
]

SYNTHETIC_CONFIG = {
    "T": 120,
    "N": 50,
    "m": 75,
    "seed": 10,
    "start_date": "1990-01-01",
    "include_intercept": False,
    "beta_rho": 0.9,
    "sigma_beta": 0.5,
    "z_rho": 0.4,
    "z_scale": 1.0,
    "heavy_tail_df": 5.0,
    "sigma_eps_base": 0.5,
    "hetero_strength": 0.5,
    "missing_prob": 0.05,
    "missing_mode": "mcAR",
    "impute": "zero",
    "z_drift_scale": 0.02,
}

FULL_SEEDS = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 110, 120]
LARGE_RFF_SEEDS = [10, 20, 30]
RFF_SMALL_COMPONENTS = [24, 32, 48, 64, 128, 256]
RFF_LARGE_COMPONENTS = [512, 1024, 2048, 4096]
BASELINE_RFF_COMPONENT = 0
RFF_GAMMA = 0.25
ALPHA = 100
SHOW_PROGRESS = True

FORECAST_START = "1993-01-01"
MIN_TRAIN_OBS = 500
MARKET_CAP_TOP_N = 50

SAVE_RESULTS_TO_PICKLES = True
LOAD_RESULTS_FROM_PICKLES = False

char_data = None
truth = None
returns_data = None
run_results = {}
analysis_outputs = {}
created_files = []
ACTIVE_EXPERIMENT = None

def normalize_data_label(label):
    if label is None:
        return None
    safe = "_".join(str(label).strip().split())
    return None if safe.lower() in {"", "all", "all_data", "real", "openap"} else safe

def build_result_pickle_name(base_name, n_factors, train_window_months, data_label=None):
    safe_base = "_".join(str(base_name).strip().split())
    label = normalize_data_label(data_label)
    parts = [safe_base]
    if label:
        parts.append(label)
    parts.extend([str(n_factors), str(train_window_months)])
    return "_".join(parts) + ".pkl"

def parse_experiment_file(path_like, base_name):
    path = Path(path_like)
    parts = path.stem.split("_")
    base_parts = base_name.split("_")
    if parts[: len(base_parts)] != base_parts or len(parts) < len(base_parts) + 2:
        raise ValueError(f"Could not parse experiment metadata from {path.name}")
    k = int(parts[-2])
    T = int(parts[-1])
    label = "_".join(parts[len(base_parts): -2]) or None
    return {"path": path, "k": k, "T": T, "data_label": label}

def discover_experiment_files(base_name, data_label=None):
    target_label = normalize_data_label(data_label)
    matches = []
    for path in sorted(RESULTS_DIR.glob(f"{base_name}*.xlsx")):
        try:
            info = parse_experiment_file(path, base_name)
        except Exception:
            continue
        file_label = normalize_data_label(info["data_label"])
        if target_label is None:
            if file_label is not None:
                continue
        elif file_label != target_label:
            continue
        matches.append(info)
    return sorted(matches, key=lambda item: (item["k"], item["T"], item["path"].name))

config_preview = pd.DataFrame(RUN_CONFIGS)
print(f"USE_SYNTHETIC_DATA={USE_SYNTHETIC_DATA}, data_label='{data_label}'")
config_preview


**Synthetic Data (Run When `USE_SYNTHETIC_DATA=True`)**

In [ ]:
from src.IPCA_Grass_estimator import generate_ipca_workflow_panel

if USE_SYNTHETIC_DATA:
    synthetic_kwargs = dict(SYNTHETIC_CONFIG)
    synthetic_kwargs["m"] = max(
        synthetic_kwargs["m"],
        max(cfg["k"] for cfg in RUN_CONFIGS),
    )
    char_data, truth = generate_ipca_workflow_panel(**synthetic_kwargs)
    print(
        f"Synthetic panel ready: {char_data.shape[0]:,} rows, "
        f"{char_data['permno'].nunique()} assets, "
        f"{char_data['yyyymm'].nunique()} months."
    )
    char_data.head()
else:
    print("Skipping synthetic generation because USE_SYNTHETIC_DATA=False.")


**OpenAP Download And Cleaning (Real-Data Path Only)**

In [ ]:
from src.data_pipeline import DataPipeline

if not USE_SYNTHETIC_DATA and not USE_CACHED_REAL_CHAR_DATA:
    new_mod = DataPipeline()
    char_data = new_mod.download_openap_data(**REAL_CHAR_DOWNLOAD)
    print(f"Downloaded OpenAP characteristics: {char_data.shape}")
else:
    print("Skipping OpenAP download.")


In [ ]:
from src.data_pipeline import DataPipeline

if not USE_SYNTHETIC_DATA and not USE_CACHED_REAL_CHAR_DATA:
    if char_data is None:
        raise ValueError("char_data is not loaded yet. Run the OpenAP download cell first.")
    new_mod = DataPipeline()
    char_data, dropped_cols = new_mod.remove_mostly_nan_columns(char_data, max_nan_frac=0.5)
    print(f"Dropped mostly-NaN columns: {dropped_cols}")
    char_data, kept_chars, dropped_chars = new_mod.drop_low_std_and_high_corr(
        char_data,
        min_std=1e-5,
        max_corr=0.75,
    )
    char_data = new_mod.fill_remaining_missing(char_data, use_past_only=True)
    print(f"Kept {len(kept_chars)} characteristics after cleaning.")
    char_data.head()
else:
    print("Skipping real-data characteristic cleaning.")


**Optional Cached Real-Data Load**

In [ ]:
if not USE_SYNTHETIC_DATA and USE_CACHED_REAL_CHAR_DATA:
    if not CHAR_DATA_CACHE_PATH.exists():
        raise FileNotFoundError(f"Could not find {CHAR_DATA_CACHE_PATH}")
    char_data = pd.read_parquet(CHAR_DATA_CACHE_PATH, engine="pyarrow")
    print(f"Loaded cached characteristics from {CHAR_DATA_CACHE_PATH}")
    char_data.head()
else:
    print("Skipping cached characteristic load.")


**WRDS Returns Merge (Real-Data Path Only)**

In [ ]:
from src.data_pipeline import DataPipeline

if not USE_SYNTHETIC_DATA:
    new_mod = DataPipeline()
    returns_data = new_mod.download_sp500_returns_wrds(**REAL_RETURNS_DOWNLOAD)
    print(f"Downloaded returns panel: {returns_data.shape}")
else:
    print("Skipping return download because the synthetic panel already contains returns.")


**Finalize Panel And Inspect**

In [ ]:
from src.data_pipeline import DataPipeline

if not USE_SYNTHETIC_DATA:
    if char_data is None:
        raise ValueError("char_data is not loaded yet.")
    if returns_data is None:
        raise ValueError("returns_data is not loaded yet.")
    new_mod = DataPipeline()
    returns_data = returns_data.rename(columns={"excess_ret": "ret_adj"})
    char_data = new_mod.merge_openap_with_crsp_returns(char_data, returns_data)
    print(f"Merged real-data panel shape: {char_data.shape}")
else:
    print("Skipping real-data merge.")


**Working Panel Preview**

In [ ]:
if char_data is None:
    raise ValueError("char_data is not ready yet. Finish either the synthetic or real-data path first.")

print(
    f"Working panel: {char_data.shape[0]:,} rows, "
    f"{char_data['permno'].nunique()} assets, "
    f"{char_data['yyyymm'].nunique()} months."
)
char_data.head()


**Run Multi-(k, T) Sweeps**

This cell loops over every `(k, T)` pair in `RUN_CONFIGS` and runs the full RFF sweep on the current `char_data` panel.

It also saves one pickle per completed experiment, so if the runs take a while you can later set `LOAD_RESULTS_FROM_PICKLES = True` and jump straight to the export and plotting cells.


In [ ]:
from src.grassmann_ipca_workflow import GrassmannIPCAWorkflow

def run_rff_sweep(char_data, k, train_window_months):
    wf = GrassmannIPCAWorkflow()
    result_rff_all = {seed: {} for seed in FULL_SEEDS}
    market_cap_col = "mcap" if "mcap" in char_data.columns else None
    market_cap_top_n = min(MARKET_CAP_TOP_N, char_data["permno"].nunique()) if market_cap_col else None

    common_kwargs = dict(
        char_data=char_data,
        forecast_start=FORECAST_START,
        target_col="y_ipca",
        n_factors=k,
        train_window_months=train_window_months,
        min_train_obs=MIN_TRAIN_OBS,
        normalize=True,
        mean_factor=True,
        silent=True,
        iter_tol=1e-4,
        warm_start=True,
        alpha=ALPHA,
        show_progress=SHOW_PROGRESS,
        market_cap_filter_col=market_cap_col,
        market_cap_filter_top_n=market_cap_top_n,
    )

    baseline_seed = FULL_SEEDS[0]
    result_rff_all[baseline_seed][BASELINE_RFF_COMPONENT] = wf.rolling_ipca_predictions(
        use_rff=False,
        rff_n_components=0,
        rff_gamma=RFF_GAMMA,
        rff_random_state=baseline_seed,
        **common_kwargs,
    )

    for seed in FULL_SEEDS:
        for rff_n_component in RFF_SMALL_COMPONENTS:
            result_rff_all[seed][rff_n_component] = wf.rolling_ipca_predictions(
                use_rff=True,
                rff_n_components=rff_n_component,
                rff_gamma=RFF_GAMMA,
                rff_random_state=seed,
                **common_kwargs,
            )

    for seed in LARGE_RFF_SEEDS:
        for rff_n_component in RFF_LARGE_COMPONENTS:
            result_rff_all[seed][rff_n_component] = wf.rolling_ipca_predictions(
                use_rff=True,
                rff_n_components=rff_n_component,
                rff_gamma=RFF_GAMMA,
                rff_random_state=seed,
                **common_kwargs,
            )

    return result_rff_all

if char_data is None:
    raise ValueError("char_data is not ready yet. Prepare the dataset first.")

run_results = {}
saved_pickle_files = []

for cfg in RUN_CONFIGS:
    k = int(cfg["k"])
    T = int(cfg["T"])
    print(f"=== Running experiment: k={k}, T={T}, data_label='{data_label}' ===")

    result_rff_all = run_rff_sweep(
        char_data=char_data,
        k=k,
        train_window_months=T,
    )

    payload = {
        "k": k,
        "T": T,
        "data_label": data_label,
        "alpha": ALPHA,
        "rff_gamma": RFF_GAMMA,
        "result_rff_all": result_rff_all,
    }
    run_results[(k, T)] = payload

    if SAVE_RESULTS_TO_PICKLES:
        out_path = RESULTS_DIR / build_result_pickle_name(
            "result_rff_all",
            k,
            T,
            data_label=data_label,
        )
        with open(out_path, "wb") as f:
            pickle.dump(payload, f)
        saved_pickle_files.append(out_path.name)
        print(f"Saved {out_path.name}")

pd.DataFrame(
    [
        {
            "k": payload["k"],
            "T": payload["T"],
            "data_label": payload["data_label"],
            "n_rff_levels": len({comp for seed_metrics in payload["result_rff_all"].values() for comp in seed_metrics}),
        }
        for payload in run_results.values()
    ]
).sort_values(["k", "T"]).reset_index(drop=True)


In [ ]:
if LOAD_RESULTS_FROM_PICKLES:
    run_results = {}
    for cfg in RUN_CONFIGS:
        k = int(cfg["k"])
        T = int(cfg["T"])
        in_path = RESULTS_DIR / build_result_pickle_name(
            "result_rff_all",
            k,
            T,
            data_label=data_label,
        )
        if not in_path.exists():
            raise FileNotFoundError(f"Missing pickle: {in_path}")
        with open(in_path, "rb") as f:
            payload = pickle.load(f)
        if "result_rff_all" not in payload:
            payload = {
                "k": k,
                "T": T,
                "data_label": data_label,
                "alpha": ALPHA,
                "rff_gamma": RFF_GAMMA,
                "result_rff_all": payload,
            }
        run_results[(k, T)] = payload

    pd.DataFrame(
        [
            {"k": payload["k"], "T": payload["T"], "data_label": payload["data_label"]}
            for payload in run_results.values()
        ]
    ).sort_values(["k", "T"]).reset_index(drop=True)
else:
    print("Skipping pickle load because LOAD_RESULTS_FROM_PICKLES=False.")


**Export OOS R², Diagnostics, And Portfolio Tables**

In [ ]:
from src.portfolio_utils import (
    build_directional_portfolio,
    build_experiment_excel_name,
    compute_portfolio_returns,
    portfolio_performance,
)

def summarize_experiment(result_rff_all):
    rff_components = sorted(
        {
            rff_n_component
            for seed_metrics in result_rff_all.values()
            for rff_n_component in seed_metrics
        }
    )
    oos_r2_results = {rff_n_component: [] for rff_n_component in rff_components}
    diag_summaries = {}
    diag_frames = []

    for seed, metrics in result_rff_all.items():
        for rff_n_component, all_metrics in metrics.items():
            pred_df_in = all_metrics[0].copy()
            diag_df_in = all_metrics[1].copy()

            ss_res = ((pred_df_in["y_true"] - pred_df_in["y_pred"]) ** 2).sum()
            ss_tot = (pred_df_in["y_true"] ** 2).sum()
            r2_oos = np.nan if ss_tot == 0 else 1 - ss_res / ss_tot
            oos_r2_results[rff_n_component].append(float(r2_oos))

            diag_df_in["seed"] = seed
            diag_df_in["rff_n_component"] = rff_n_component
            diag_frames.append(diag_df_in)
            diag_summaries[(seed, rff_n_component)] = (
                diag_df_in.describe(include="all")
                .T.reset_index()
                .rename(columns={"index": "column"})
            )

    oos_r2_avg = {
        rff_n_component: float(np.nanmean(values)) if len(values) else np.nan
        for rff_n_component, values in oos_r2_results.items()
    }
    n_runs = {
        rff_n_component: int(len(values))
        for rff_n_component, values in oos_r2_results.items()
    }
    oos_r2_export = (
        pd.DataFrame({"rff_n_component": sorted(oos_r2_avg.keys())})
        .assign(
            oos_r2=lambda d: d["rff_n_component"].map(oos_r2_avg),
            n_runs=lambda d: d["rff_n_component"].map(n_runs),
        )
        .sort_values("rff_n_component")
        .reset_index(drop=True)
    )

    features_data_df = (
        pd.concat(diag_summaries.values(), keys=diag_summaries.keys())
        .reset_index(level=[0, 1])
        .rename(columns={"level_0": "seed", "level_1": "rff_n_component"})
    )
    num_cols = [
        c for c in ["count", "mean", "min", "25%", "50%", "75%", "max", "std"]
        if c in features_data_df.columns
    ]
    features_data_df[num_cols] = features_data_df[num_cols].apply(pd.to_numeric, errors="coerce")
    features_data_df = (
        features_data_df
        .groupby(["rff_n_component", "column"], as_index=False)[num_cols]
        .mean()
    )

    mean_prediction_by_rff = {}
    portfolio_results_mean_seed = {}

    for rff_n_component in rff_components:
        seed_pred_list = []
        for seed, seed_metrics in result_rff_all.items():
            if rff_n_component not in seed_metrics:
                continue
            pred_df = seed_metrics[rff_n_component][0].copy()
            pred_df["yyyymm"] = pd.to_datetime(pred_df["yyyymm"])
            pred_df["seed"] = seed
            seed_pred_list.append(
                pred_df[
                    [
                        "permno",
                        "yyyymm",
                        "y_true",
                        "y_pred",
                        "train_end",
                        "n_train",
                        "n_test",
                        "seed",
                    ]
                ]
            )

        if not seed_pred_list:
            continue

        stacked = pd.concat(seed_pred_list, ignore_index=True)
        y_true_check = stacked.groupby(["permno", "yyyymm"])["y_true"].nunique()
        if (y_true_check > 1).any():
            raise ValueError(
                f"y_true is not identical across seeds for rff_n_component={rff_n_component}"
            )

        mean_pred_df = (
            stacked
            .groupby(["permno", "yyyymm"], as_index=False)
            .agg(
                y_true=("y_true", "first"),
                y_pred=("y_pred", "mean"),
                train_end=("train_end", "first"),
                n_train=("n_train", "first"),
                n_test=("n_test", "first"),
                n_seeds=("seed", "nunique"),
            )
            .sort_values(["yyyymm", "permno"])
            .reset_index(drop=True)
        )

        mean_prediction_by_rff[rff_n_component] = mean_pred_df
        port = build_directional_portfolio(mean_pred_df)
        monthly_ret = compute_portfolio_returns(port)
        ss_res = ((mean_pred_df["y_true"] - mean_pred_df["y_pred"]) ** 2).sum()
        ss_tot = (mean_pred_df["y_true"] ** 2).sum()
        oos_r2 = np.nan if ss_tot == 0 else 1 - ss_res / ss_tot
        stats = portfolio_performance(monthly_ret)
        stats["oos_r2"] = float(oos_r2) if pd.notna(oos_r2) else np.nan

        portfolio_results_mean_seed[rff_n_component] = {
            "mean_pred_df": mean_pred_df,
            "portfolio": port,
            "returns": monthly_ret,
            "oos_r2": oos_r2,
            "performance": stats,
        }

    perf_table = pd.DataFrame(
        {k: v["performance"] for k, v in portfolio_results_mean_seed.items()}
    ).T.sort_index()
    perf_table.index.name = "rff_n_component"

    diag_stack = pd.concat(diag_frames, ignore_index=True) if diag_frames else pd.DataFrame()

    return {
        "oos_r2_results": oos_r2_results,
        "oos_r2_avg": oos_r2_avg,
        "oos_r2_export": oos_r2_export,
        "features_data_df": features_data_df,
        "mean_prediction_by_rff": mean_prediction_by_rff,
        "portfolio_results_mean_seed": portfolio_results_mean_seed,
        "perf_table": perf_table,
        "diag_stack": diag_stack,
    }

if not run_results:
    raise ValueError("run_results is empty. Run the sweep cell or load the saved pickles first.")

analysis_outputs = {}
created_files = []

for (k, T), payload in sorted(run_results.items()):
    summary = summarize_experiment(payload["result_rff_all"])
    analysis_outputs[(k, T)] = summary

    features_export_name = build_experiment_excel_name(
        "features_data",
        k,
        T,
        data_label=payload["data_label"],
    )
    portfolio_export_name = build_experiment_excel_name(
        "portfolio_performance",
        k,
        T,
        data_label=payload["data_label"],
    )
    oos_r2_export_name = build_experiment_excel_name(
        "oos_r2",
        k,
        T,
        data_label=payload["data_label"],
    )

    summary["features_data_df"].to_excel(RESULTS_DIR / features_export_name, index=False)
    summary["perf_table"].to_excel(RESULTS_DIR / portfolio_export_name, index=True)
    summary["oos_r2_export"].assign(k=k, T=T).to_excel(
        RESULTS_DIR / oos_r2_export_name,
        index=False,
    )

    created_files.extend(
        [
            {"artifact": "features", "file": features_export_name, "k": k, "T": T},
            {"artifact": "portfolio", "file": portfolio_export_name, "k": k, "T": T},
            {"artifact": "oos_r2", "file": oos_r2_export_name, "k": k, "T": T},
        ]
    )

ACTIVE_EXPERIMENT = (RUN_CONFIGS[-1]["k"], RUN_CONFIGS[-1]["T"])
if ACTIVE_EXPERIMENT not in analysis_outputs:
    ACTIVE_EXPERIMENT = sorted(analysis_outputs.keys())[0]

pd.DataFrame(created_files).sort_values(["artifact", "k", "T"]).reset_index(drop=True)


In [ ]:
analysis_outputs[ACTIVE_EXPERIMENT]["oos_r2_export"]

In [ ]:
analysis_outputs[ACTIVE_EXPERIMENT]["features_data_df"].head(20)

In [ ]:
analysis_outputs[ACTIVE_EXPERIMENT]["perf_table"]

In [ ]:
pd.DataFrame(created_files).sort_values(["artifact", "k", "T"]).reset_index(drop=True)

**Single-Experiment OOS R² Diagnostic Plot**

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

active_summary = analysis_outputs[ACTIVE_EXPERIMENT]
k_active, T_active = ACTIVE_EXPERIMENT
oos_r2_avg = active_summary["oos_r2_avg"]
diag_stack = active_summary["diag_stack"]
n_train = int(diag_stack["n_train"].median()) if not diag_stack.empty else 1

m_vals = np.array(sorted(oos_r2_avg.keys()))
r2_vals = np.array([100 * oos_r2_avg[m] for m in m_vals])
x_vals = 2 * m_vals * k_active / max(n_train, 1)

mask_zero = m_vals == 0
mask_nonzero = m_vals > 0

x_zero = x_vals[mask_zero]
y_zero = r2_vals[mask_zero]
x_nonzero = x_vals[mask_nonzero]
y_nonzero = r2_vals[mask_nonzero]
m_nonzero = m_vals[mask_nonzero]

positive_idx = np.where(y_nonzero > 0)[0]
crossover_x = x_nonzero[positive_idx[0]] if len(positive_idx) else None
crossover_y = y_nonzero[positive_idx[0]] if len(positive_idx) else None

plt.style.use("seaborn-v0_8-whitegrid")
fig, (ax_top, ax_bot) = plt.subplots(
    2,
    1,
    figsize=(11, 7),
    sharex=True,
    gridspec_kw={"height_ratios": [1, 4], "hspace": 0.05},
)

line_color = "#e53935"
threshold_color = "#9575cd"
under_color = "#a5d6a7"
over_color = "#ef9a9a"
xmax = max(1.1, x_vals.max() * 1.1)

for ax in (ax_top, ax_bot):
    ax.axvspan(0, 1, color=under_color, alpha=0.10, lw=0)
    ax.axvspan(1, xmax, color=over_color, alpha=0.10, lw=0)
    ax.axvline(1, color=threshold_color, linestyle="--", linewidth=1.8, alpha=0.95)
    ax.grid(True, alpha=0.25)

if len(x_zero):
    ax_top.plot(x_zero, y_zero, marker="X", markersize=14, color=line_color, linewidth=0)
    ax_top.annotate(
        fr"$\tilde{{m}}=0$: {y_zero[0]:.2f}%",
        xy=(x_zero[0], y_zero[0]),
        xytext=(10, -2),
        textcoords="offset points",
        color=line_color,
        fontsize=13,
        va="center",
    )

ax_bot.plot(
    x_nonzero,
    y_nonzero,
    color=line_color,
    linewidth=3,
    marker="D",
    markersize=10,
    markerfacecolor="white",
    markeredgecolor=line_color,
    markeredgewidth=2.5,
)

for m_label in [24, 64, 256, 1024, 4096]:
    if m_label in oos_r2_avg and m_label in m_nonzero:
        i = np.where(m_nonzero == m_label)[0][0]
        ax_bot.annotate(
            fr"$\tilde{{m}}={m_label}$",
            xy=(x_nonzero[i], y_nonzero[i]),
            xytext=(8, 8),
            textcoords="offset points",
            fontsize=12,
            bbox=dict(boxstyle="round,pad=0.18", fc="white", ec="none", alpha=0.7),
        )

if crossover_x is not None:
    ax_bot.annotate(
        fr"crossover at $p/n \approx {crossover_x:.2f}$",
        xy=(crossover_x, crossover_y),
        xytext=(crossover_x + 0.16 * xmax, crossover_y + 0.15),
        color=threshold_color,
        fontsize=14,
        arrowprops=dict(arrowstyle="-", color=threshold_color, lw=1.8, alpha=0.75),
    )

ax_top.set_ylim(-4.0, -2.0)
ax_bot.set_ylim(min(0.0, y_nonzero.min() - 0.1 if len(y_nonzero) else 0.0), max(1.05, y_nonzero.max() + 0.1 if len(y_nonzero) else 1.05))
ax_bot.set_xlim(-0.5, xmax)

ax_top.spines["bottom"].set_visible(False)
ax_bot.spines["top"].set_visible(False)
ax_top.tick_params(labeltop=False, bottom=False)
ax_bot.xaxis.tick_bottom()

d = 0.015
kwargs = dict(transform=ax_top.transAxes, color="k", clip_on=False, linewidth=1.2)
ax_top.plot((-d, +d), (-d, +d), **kwargs)
ax_top.plot((1 - d, 1 + d), (-d, +d), **kwargs)
kwargs.update(transform=ax_bot.transAxes)
ax_bot.plot((-d, +d), (1 - d, 1 + d), **kwargs)
ax_bot.plot((1 - d, 1 + d), (1 - d, 1 + d), **kwargs)

ax_top.set_ylabel(r"OOS $R^2$ (%)", fontsize=16)
ax_bot.set_ylabel(r"OOS $R^2$ (%)", fontsize=16)
ax_bot.set_xlabel(r"$p/n = 2\tilde{m}k / n_{\mathrm{train}}$", fontsize=16)

fig.suptitle(
    fr"OOS $R^2$ crossover: $k={k_active}, T={T_active}, \alpha={ALPHA}$",
    fontsize=20,
    y=0.98,
)

legend_handles = [
    Patch(facecolor=under_color, alpha=0.10, label="under-parameterized"),
    Patch(facecolor=over_color, alpha=0.10, label="over-parameterized"),
    Line2D([0], [0], color=threshold_color, lw=1.8, linestyle="--", label=r"interpolation threshold $p/n=1$"),
]
ax_bot.legend(handles=legend_handles, loc="lower right", fontsize=12, frameon=True)

plt.tight_layout()
plt.show()


**Comparison plots from exported Excel files**

Run the export cell first so the `features_data*.xlsx`, `portfolio_performance*.xlsx`, and `oos_r2*.xlsx` files exist for every experiment.


In [ ]:
import matplotlib.pyplot as plt

feature_infos = discover_experiment_files("features_data", data_label=PLOT_DATA_LABEL)
if not feature_infos:
    raise FileNotFoundError("No features_data Excel files found for the selected data label.")

feature_files = [info["path"] for info in feature_infos]
styles = [
    dict(color="#1f77b4", marker="o", linestyle="-"),
    dict(color="#2ca02c", marker="s", linestyle="--"),
    dict(color="#9467bd", marker="^", linestyle="-"),
    dict(color="#ff7f0e", marker="v", linestyle=":"),
    dict(color="#d62728", marker="D", linestyle="-"),
    dict(color="#8c564b", marker="P", linestyle="--"),
    dict(color="#e377c2", marker="X", linestyle="-."),
]

def parse_k_t(file_name):
    info = parse_experiment_file(file_name, "features_data")
    return info["k"], info["T"]

def load_metric(file_name, metric_names):
    df = pd.read_excel(file_name)
    df.columns = [str(c).strip() for c in df.columns]
    need = {"rff_n_component", "column", "mean", "std"}
    missing = need - set(df.columns)
    if missing:
        raise ValueError(f"{file_name} is missing columns: {missing}")

    out = df[df["column"].astype(str).str.strip().isin(metric_names)].copy()
    if out.empty:
        raise ValueError(f"{file_name} does not contain any of {metric_names}")

    out["rff_n_component"] = pd.to_numeric(out["rff_n_component"], errors="coerce")
    out["mean"] = pd.to_numeric(out["mean"], errors="coerce")
    out["std"] = pd.to_numeric(out["std"], errors="coerce").fillna(0.0)
    out = (
        out.dropna(subset=["rff_n_component", "mean"])
        .groupby("rff_n_component", as_index=False)[["mean", "std"]]
        .mean()
        .sort_values("rff_n_component")
    )
    return out[out["rff_n_component"] > 0].copy()

plt.style.use("seaborn-v0_8-whitegrid")
fig, axes = plt.subplots(2, 2, figsize=(16, 10), constrained_layout=True)

for i, file_name in enumerate(feature_files):
    style = styles[i % len(styles)]
    k, T = parse_k_t(file_name)
    label = fr"$k={k},\, T={T}$"

    erank_df = load_metric(file_name, {"erank", "effective_rank"})
    gap_df = load_metric(file_name, {"gap_ratio"})
    grass_df = load_metric(file_name, {"grassmann_dist"})
    angle_df = load_metric(file_name, {"principal_angle_max"})

    axes[0, 0].plot(erank_df["rff_n_component"], erank_df["mean"], label=label, linewidth=2.6, markersize=7.5, **style)
    axes[0, 0].fill_between(erank_df["rff_n_component"], erank_df["mean"] - erank_df["std"], erank_df["mean"] + erank_df["std"], color=style["color"], alpha=0.10)

    axes[0, 1].plot(gap_df["rff_n_component"], gap_df["mean"], label=label, linewidth=2.6, markersize=7.5, **style)
    axes[0, 1].fill_between(gap_df["rff_n_component"], gap_df["mean"] - gap_df["std"], gap_df["mean"] + gap_df["std"], color=style["color"], alpha=0.10)

    axes[1, 0].plot(grass_df["rff_n_component"], grass_df["mean"], label=label, linewidth=2.6, markersize=7.5, **style)
    axes[1, 0].fill_between(grass_df["rff_n_component"], grass_df["mean"] - grass_df["std"], grass_df["mean"] + grass_df["std"], color=style["color"], alpha=0.10)

    axes[1, 1].plot(angle_df["rff_n_component"], angle_df["mean"], label=label, linewidth=2.6, markersize=7.5, **style)
    axes[1, 1].fill_between(angle_df["rff_n_component"], angle_df["mean"] - angle_df["std"], angle_df["mean"] + angle_df["std"], color=style["color"], alpha=0.10)

for ax in axes[0]:
    ax.set_xscale("log")
    ax.set_xlabel(r"RFF components $\tilde{m}$ (log scale)", fontsize=13)
    ax.grid(True, alpha=0.25)

for ax in axes[1]:
    ax.set_xlabel(r"RFF components $\tilde{m}$", fontsize=13)
    ax.grid(True, alpha=0.25)

axes[0, 0].set_title("Effective rank vs. RFF (log scale)", fontsize=16)
axes[0, 0].set_ylabel(r"$\mathrm{erank}(\hat{\Sigma}_t)$", fontsize=13)
axes[0, 1].set_title("Gap ratio vs. RFF (log scale)", fontsize=16)
axes[0, 1].set_ylabel(r"Gap ratio $\lambda_{\min}/\lambda_{\max}$", fontsize=13)
axes[1, 0].set_title("Subspace stability vs. RFF", fontsize=16)
axes[1, 0].set_ylabel("Geodesic Grassmann distance", fontsize=13)
axes[1, 1].set_title("Max principal angle vs. RFF", fontsize=16)
axes[1, 1].set_ylabel("Max principal angle (rad)", fontsize=13)
axes[0, 1].legend(frameon=True, fontsize=11, loc="best")
axes[1, 1].legend(frameon=True, fontsize=11, loc="best")

plt.show()


In [ ]:
import re
import xml.etree.ElementTree as ET
import zipfile
import matplotlib.pyplot as plt

portfolio_infos = discover_experiment_files("portfolio_performance", data_label=PLOT_DATA_LABEL)
if not portfolio_infos:
    raise FileNotFoundError("No portfolio_performance Excel files found for the selected data label.")

portfolio_files = [info["path"] for info in portfolio_infos]
styles = [
    dict(color="#1f77b4", marker="o", linestyle="-"),
    dict(color="#2ca02c", marker="s", linestyle="--"),
    dict(color="#9467bd", marker="^", linestyle="-"),
    dict(color="#ff7f0e", marker="v", linestyle=":"),
    dict(color="#d62728", marker="D", linestyle="-"),
]

def excel_col_to_idx(col_letters):
    n = 0
    for ch in col_letters:
        n = n * 26 + (ord(ch.upper()) - ord("A") + 1)
    return n - 1

def read_xlsx_fallback(path):
    ns = {"a": "http://schemas.openxmlformats.org/spreadsheetml/2006/main"}
    with zipfile.ZipFile(path) as zf:
        shared = []
        if "xl/sharedStrings.xml" in zf.namelist():
            root = ET.fromstring(zf.read("xl/sharedStrings.xml"))
            for si in root.findall("a:si", ns):
                shared.append("".join(t.text or "" for t in si.findall(".//a:t", ns)))

        root = ET.fromstring(zf.read("xl/worksheets/sheet1.xml"))
        rows = []
        for row in root.findall(".//a:sheetData/a:row", ns):
            cell_map = {}
            max_idx = -1
            for c in row.findall("a:c", ns):
                ref = c.attrib.get("r", "")
                m = re.match(r"([A-Z]+)", ref)
                if not m:
                    continue
                idx = excel_col_to_idx(m.group(1))
                max_idx = max(max_idx, idx)
                cell_type = c.attrib.get("t")
                if cell_type == "inlineStr":
                    is_node = c.find("a:is", ns)
                    val = "" if is_node is None else "".join(t.text or "" for t in is_node.findall(".//a:t", ns))
                else:
                    v = c.find("a:v", ns)
                    val = "" if v is None or v.text is None else v.text
                    if cell_type == "s" and val != "":
                        val = shared[int(val)]
                cell_map[idx] = val
            if max_idx >= 0:
                vals = [""] * (max_idx + 1)
                for idx, val in cell_map.items():
                    vals[idx] = val
                rows.append(vals)

    max_cols = max(len(r) for r in rows)
    rows = [r + [""] * (max_cols - len(r)) for r in rows]
    header = rows[0]
    header = ["rff_n_component" if i == 0 and str(h).strip() == "" else str(h).strip() for i, h in enumerate(header)]
    return pd.DataFrame(rows[1:], columns=header)

def read_portfolio_excel(path):
    try:
        df = pd.read_excel(path)
    except Exception:
        df = read_xlsx_fallback(path)

    df.columns = [str(c).strip() for c in df.columns]
    if "rff_n_component" not in df.columns:
        first_col = df.columns[0]
        df = df.rename(columns={first_col: "rff_n_component"})
    for col in ["rff_n_component", "sharpe", "max_dd"]:
        if col not in df.columns:
            raise ValueError(f"{path} is missing required column: {col}")
        df[col] = pd.to_numeric(df[col], errors="coerce")
    return df.dropna(subset=["rff_n_component", "sharpe", "max_dd"]).sort_values("rff_n_component")

plt.style.use("seaborn-v0_8-whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(15, 5), constrained_layout=True)
best_sharpe = (-np.inf, None, None, None)

for i, path in enumerate(portfolio_files):
    style = styles[i % len(styles)]
    df = read_portfolio_excel(path)
    info = parse_experiment_file(path, "portfolio_performance")
    k, T = info["k"], info["T"]
    label = fr"$k={k},\, T={T}$"

    axes[0].plot(df["rff_n_component"], df["sharpe"], label=label, linewidth=2.4, markersize=7, **style)
    axes[1].plot(df["rff_n_component"], 100 * df["max_dd"].abs(), label=label, linewidth=2.4, markersize=7, **style)

    i_best = df["sharpe"].idxmax()
    sharpe_peak = float(df.loc[i_best, "sharpe"])
    m_peak = int(df.loc[i_best, "rff_n_component"])
    if sharpe_peak > best_sharpe[0]:
        best_sharpe = (sharpe_peak, m_peak, style["color"], label)

axes[0].set_title("Sharpe vs. RFF", fontsize=17)
axes[0].set_xlabel(r"RFF components $\tilde{m}$", fontsize=13)
axes[0].set_ylabel("Sharpe ratio", fontsize=13)
axes[0].grid(True, alpha=0.25)
axes[0].legend(frameon=True, fontsize=11, loc="best")

if best_sharpe[1] is not None:
    y_peak, m_peak, color_peak, _ = best_sharpe
    axes[0].annotate(
        fr"Sharpe peak: $\tilde{{m}}={m_peak}$, $\approx {y_peak:.2f}$",
        xy=(m_peak, y_peak),
        xytext=(10, 10),
        textcoords="offset points",
        color=color_peak,
        fontsize=12,
        arrowprops=dict(arrowstyle="-", color=color_peak, lw=1.5),
    )

axes[1].set_title("Max drawdown vs. RFF", fontsize=17)
axes[1].set_xlabel(r"RFF components $\tilde{m}$", fontsize=13)
axes[1].set_ylabel("Max drawdown (%)", fontsize=13)
axes[1].grid(True, alpha=0.25)
axes[1].invert_yaxis()
axes[1].legend(frameon=True, fontsize=11, loc="best")

plt.show()


In [ ]:
pd.DataFrame(created_files).sort_values(["artifact", "k", "T"]).reset_index(drop=True)

In [ ]:
import matplotlib.pyplot as plt
import re
import xml.etree.ElementTree as ET
import zipfile

r2_infos = discover_experiment_files("oos_r2", data_label=PLOT_DATA_LABEL)
if not r2_infos:
    raise FileNotFoundError("No oos_r2 Excel files found for the selected data label.")

r2_files = [info["path"] for info in r2_infos]
styles = {
    (12, 24): dict(color="#1f77b4", marker="o", linestyle="-", linewidth=3.2, markersize=13),
    (14, 36): dict(color="#2ca02c", marker="s", linestyle="--", linewidth=3.2, markersize=13),
    (16, 24): dict(color="#9467bd", marker="^", linestyle="-", linewidth=3.2, markersize=13),
    (18, 24): dict(color="#ff7f0e", marker="v", linestyle=":", linewidth=3.2, markersize=13),
    (18, 36): dict(color="#d62728", marker="D", linestyle="-", linewidth=3.6, markersize=13),
}

def excel_col_to_idx(col_letters):
    n = 0
    for ch in col_letters:
        n = n * 26 + (ord(ch.upper()) - ord("A") + 1)
    return n - 1

def read_xlsx_fallback(path):
    ns = {"a": "http://schemas.openxmlformats.org/spreadsheetml/2006/main"}
    with zipfile.ZipFile(path) as zf:
        shared = []
        if "xl/sharedStrings.xml" in zf.namelist():
            root = ET.fromstring(zf.read("xl/sharedStrings.xml"))
            for si in root.findall("a:si", ns):
                shared.append("".join(t.text or "" for t in si.findall(".//a:t", ns)))

        root = ET.fromstring(zf.read("xl/worksheets/sheet1.xml"))
        rows = []
        for row in root.findall(".//a:sheetData/a:row", ns):
            cell_map = {}
            max_idx = -1
            for c in row.findall("a:c", ns):
                ref = c.attrib.get("r", "")
                m = re.match(r"([A-Z]+)", ref)
                if not m:
                    continue
                idx = excel_col_to_idx(m.group(1))
                max_idx = max(max_idx, idx)
                cell_type = c.attrib.get("t")
                if cell_type == "inlineStr":
                    is_node = c.find("a:is", ns)
                    val = "" if is_node is None else "".join(t.text or "" for t in is_node.findall(".//a:t", ns))
                else:
                    v = c.find("a:v", ns)
                    val = "" if v is None or v.text is None else v.text
                    if cell_type == "s" and val != "":
                        val = shared[int(val)]
                cell_map[idx] = val
            if max_idx >= 0:
                vals = [""] * (max_idx + 1)
                for idx, val in cell_map.items():
                    vals[idx] = val
                rows.append(vals)

    max_cols = max(len(r) for r in rows)
    rows = [r + [""] * (max_cols - len(r)) for r in rows]
    return pd.DataFrame(rows[1:], columns=[str(x).strip() for x in rows[0]])

def read_r2_excel(path):
    try:
        df = pd.read_excel(path)
    except Exception:
        df = read_xlsx_fallback(path)
    df.columns = [str(c).strip() for c in df.columns]
    df["rff_n_component"] = pd.to_numeric(df["rff_n_component"], errors="coerce")
    df["oos_r2"] = pd.to_numeric(df["oos_r2"], errors="coerce")
    return df.dropna(subset=["rff_n_component", "oos_r2"]).sort_values("rff_n_component").reset_index(drop=True)

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "font.family": "serif",
    "mathtext.fontset": "cm",
    "axes.titlesize": 24,
    "axes.labelsize": 22,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
    "legend.fontsize": 19,
})

fig, ax = plt.subplots(figsize=(8.6, 9.0), dpi=150)
best = (-np.inf, None, None, None)

fallback_styles = [
    dict(color="#1f77b4", marker="o", linestyle="-", linewidth=3.2, markersize=13),
    dict(color="#2ca02c", marker="s", linestyle="--", linewidth=3.2, markersize=13),
    dict(color="#9467bd", marker="^", linestyle="-", linewidth=3.2, markersize=13),
    dict(color="#ff7f0e", marker="v", linestyle=":", linewidth=3.2, markersize=13),
    dict(color="#d62728", marker="D", linestyle="-", linewidth=3.6, markersize=13),
    dict(color="#8c564b", marker="P", linestyle="--", linewidth=3.2, markersize=13),
]

for i, file_name in enumerate(r2_files):
    info = parse_experiment_file(file_name, "oos_r2")
    k, T = info["k"], info["T"]
    df = read_r2_excel(file_name)
    y_pct = 100.0 * df["oos_r2"].to_numpy()
    style = styles.get((k, T), fallback_styles[i % len(fallback_styles)])

    ax.plot(df["rff_n_component"], y_pct, label=fr"$k={k},\, T={T}$", **style)

    local_idx = int(np.argmax(y_pct))
    local_best = y_pct[local_idx]
    local_x = float(df.iloc[local_idx]["rff_n_component"])
    if local_best > best[0]:
        best = (local_best, local_x, style["color"], (k, T))

ax.set_title(r"OOS $R^2$ vs. RFF", pad=14)
ax.set_xlabel(r"RFF components $P$")
ax.set_ylabel(r"OOS $R^2$ (%)")
ax.grid(True, alpha=0.25, color="#b0b0b0")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_linewidth(1.7)
ax.spines["bottom"].set_linewidth(1.7)
ax.tick_params(width=1.5, length=7)

if best[1] is not None:
    best_y, best_x, best_color, (best_k, best_T) = best
    ax.annotate(
        fr"best: $k={best_k}, T={best_T}$, {best_y:.2f}%",
        xy=(best_x, best_y),
        xytext=(700, best_y + 0.02),
        color=best_color,
        fontsize=18,
        arrowprops=dict(arrowstyle="-", lw=1.8, color=best_color, alpha=0.75),
    )

leg = ax.legend(loc="best", frameon=False, handlelength=1.8)
for h in leg.get_lines():
    h.set_linewidth(3.5)

plt.tight_layout()
plt.show()
